# Lab 53: Cost and latency observability

The [observability guide](../../concepts/observability/observability-for-agent-pms.md) put cost per session (mean and p90/p99) on the PM's metric list. Make it measurable: account per-session cost across tokens, runtime, and tool calls; surface the tail; detect runaway loops; quantify re-sent context; and simulate model routing. Fill in the `TODO` cells; reference in `solution/`.

## Step 0: Setup

In [ ]:
import json
from cost import (load_sessions, cost_summary, session_cost, detect_runaways,
                  resent_context_fraction, with_caching, route_cheaper)
sessions = load_sessions()
print(f"{len(sessions)} session traces (tokens + runtime + tool calls per step)")

## Step 1: The tail dominates (own p90/p99, not the mean)

In [ ]:
# TODO: call cost_summary(sessions) and print mean, p50, p90, p99, max. How many times the
# mean is the p99? What would a mean-only dashboard miss?
raise NotImplementedError

## Step 2: Runaway-loop detection

In [ ]:
# Runaway detection: a session that calls the same (tool, args) over and over is stuck in a
# loop - the pattern behind the weekend-long sessions that burn thousands of dollars.
flagged = detect_runaways(sessions, loop_threshold=10)
print("runaway sessions:", flagged)
for sid in flagged:
    sess = next(x for x in sessions if x["id"]==sid)
    print(f"  {sid}: {len(sess['steps'])} steps, ${session_cost(sess):.2f}, "
          f"same query repeated {len(sess['steps'])}x")
print("\nThese are not a routing problem - they are a control-flow problem. The fix is a loop")
print("guard / step budget that kills the session, not a cheaper model.")

## Step 3: Re-sent context and caching

In [ ]:
# Re-sent context: each step re-sends the prior turns, so much of the bill is the model
# re-reading what it already saw. On normal sessions this is the ~60% the literature reports;
# the long loops push it higher.
normal = [x for x in sessions if not x.get("is_runaway")]
print(f"re-sent context: {resent_context_fraction(normal):.0%} of input tokens (normal sessions)")
print(f"                 {resent_context_fraction(sessions):.0%} overall (loops re-send the most)")
cached = with_caching(sessions, cache_frac=0.8, cache_mult=0.1)
total = cost_summary(sessions)["total"]
print(f"prompt caching 80% of history at 0.1x: total ${total:.0f} -> ${cached:.0f} "
      f"({1-cached/total:.0%} off)")

## Step 4: Model routing

In [ ]:
# TODO: on the normal (non-runaway) sessions, compare the total cost to route_cheaper(...).
# Report the saving. Why does routing NOT help the runaway sessions?
raise NotImplementedError

## Step 5: A budget gate on the tail

In [ ]:
# A budget gate: alert when the cost TAIL breaches the budget. This is the cost pillar wired
# into the same alert path as the quality signals (notify.py from Lab 41).
budget = 1.00
s = cost_summary(sessions)
print(f"mean ${s['mean']:.2f} and p90 ${s['p90']:.2f} both look within a ${budget:.2f} budget...")
print(f"...but p99 ${s['p99']:.2f}: {'BREACH - page on-call' if s['p99']>budget else 'within budget'}")
print("\nA budget set on the mean (or even p90) never fires until the runaways are already in")
print("the bill. Own the p99 tail - that is where the spend hides.")

## What you built

The cost/latency half of observability, in code: per-session cost across all three dimensions agents bill on at once (tokens, runtime, tool calls); the tail (p90/p99) that the mean hides; a runaway-loop detector; the re-sent-context share of the bill; and a model-routing simulation. The two findings that matter: **the distribution is heavy-tailed**, so you own the p90/p99 not the mean (here p99 is ~10x the mean), and **there are two distinct cost problems** - routine token spend, fixed by routing routine steps to a cheaper model (~47% off the routine bill here), and runaway loops, fixed by loop detection, not routing.

**Where this simplifies:** the session traces are deterministic synthetic data with small token counts so the lab runs offline and the arithmetic is checkable - real agent sessions are tens to hundreds of thousands of tokens, and the *ratios* (tail multiple, re-sent share, routing saving) are the lesson, not the dollar amounts. The rates are 2026 list prices and move fast - verify before quoting. Routing here uses a `simple` flag that a real system has to *predict* (a small classifier on the step), and a wrong route can cost quality, so the routing decision needs its own eval. Prompt caching is modeled as a flat multiplier; real caching has TTLs and a write premium.

With [Lab 52](../52-red-teaming-trajectories/), this closes the capability set the [observability guide](../../concepts/observability/observability-for-agent-pms.md) laid out: every pillar has a lab, and cost - the metric the guide put on the PM - is now measurable per session.